> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🤖 선형 회귀 (Linear Regression)

**회귀(숫자 예측) · 지도학습** — 몸무게 = w × 날개길이 + b

    x̄, ȳ ← 평균
    반복: 각 펭귄  dx ← x − x̄,  dy ← y − ȳ,  위합 += dx×dy,  아래합 += dx×dx
    w ← 위합 ÷ 아래합,   b ← ȳ − w × x̄

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추(또는 **Shift + Enter**)를 누르면 그 셀이 실행돼요.
> 위에서부터 **차례로** 실행하세요. 고쳐 보고 싶으면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 것으로 만드세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명과 사이트의 의사코드를 짝지어 읽으면 돼요.

- 깨끗한 펭귄 데이터를 불러와요.

In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_clean.csv')

## 1. 사이트와 같은 10마리로 직접 계산 (최소제곱법)

- zip(x, y)는 두 리스트에서 짝을 하나씩 꺼내요.
- += 는 "더해서 다시 넣기"예요(위합 ← 위합 + dx×dy).

In [ ]:
ids = [21, 31, 1, 280, 6, 278, 282, 161, 160, 154]
small = df[df['번호'].isin(ids)]
x = small['날개길이'].tolist()
y = small['몸무게'].tolist()

xb = sum(x) / len(x)
yb = sum(y) / len(y)
num = 0      # 위합
den = 0      # 아래합
for xi, yi in zip(x, y):
    dx, dy = xi - xb, yi - yb
    num += dx * dy
    den += dx * dx
w = num / den
b = yb - w * xb
print(f'w = {w:.2f},  b = {b:.2f}')
print(f'날개 210mm → {w * 210 + b:.0f}g')

- coef_가 기울기 w, intercept_가 절편 b예요. 직접 계산한 값과 똑같아요!

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(small[['날개길이']], small['몸무게'])
print(f'sklearn: w = {model.coef_[0]:.2f},  b = {model.intercept_:.2f}')

## 2. 전체 데이터로 학습·평가

예측은 숫자라서 "맞았다/틀렸다" 대신 **오차**로 평가해요. 평균 제곱 오차(MSE)와, 오차를 g 단위로 되돌린 RMSE를 봐요.

- RMSE는 MSE의 제곱근이에요. "예측이 보통 몇 g쯤 빗나가나"로 읽으면 돼요.
- score()는 R²(결정계수)를 줘요.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(df[['날개길이']], df['몸무게'], test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)
mse = mean_squared_error(y_test, pred)
print(f'w = {model.coef_[0]:.2f}, b = {model.intercept_:.1f}')
print(f'MSE = {mse:.0f},  RMSE = {mse ** 0.5:.0f}g  (보통 이만큼 빗나가요)')
print('R² 점수 =', round(model.score(X_test, y_test), 3), '(1에 가까울수록 잘 맞음)')

- 그래프에 한글(제목·축 이름)이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번 1~2분 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- 점들 사이를 지나는 빨간 직선이 모델이에요.

In [ ]:
plt.scatter(df['날개길이'], df['몸무게'], alpha=0.5, label='펭귄')
xs = pd.DataFrame({'날개길이': [170, 235]})
plt.plot(xs['날개길이'], model.predict(xs), color='red', linewidth=3, label='회귀 직선')
plt.xlabel('날개길이 (mm)'); plt.ylabel('몸무게 (g)'); plt.legend()
plt.show()

## 🚀 도전 — 경사 하강법으로 조금씩 고치기

공식 대신, 오차가 줄어드는 쪽으로 w·b를 조금씩 고쳐도 같은 직선에 다가가요(사이트 ① 쪽의 "조금씩 고치기").

- 반복할수록 MSE가 줄어들어요. 이것이 신경망이 학습하는 기본 방법이기도 해요.

In [ ]:
xs = (df['날개길이'] - df['날개길이'].mean()) / df['날개길이'].std()   # 크기 맞추기(표준화)
ys = df['몸무게']
a, c = 0.0, 0.0          # 표준화한 x에서의 기울기·절편
for step in range(200):
    err = a * xs + c - ys
    a -= 0.05 * 2 * (err * xs).mean()   # 오차가 줄어드는 쪽으로
    c -= 0.05 * 2 * err.mean()
    if step % 40 == 0:
        print(f'{step:3d}번째: MSE = {(err ** 2).mean():,.0f}')